In [1]:
import os
os.chdir('/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21')
print(os.listdir('.'))

['run_parallel2.py', 'memory', '-PROMPT-v6-DATASET.md', '.prompts', '.config', '.kernel_llm_logs_1.txt', 'ldh_worker.py']


In [2]:
import multiprocessing as mp
print("CPU count:", mp.cpu_count())

CPU count: 8


In [3]:
import os
# Check if the prior path exists
prior = "/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481"
print("prior exists?", os.path.exists(prior))
if os.path.exists(prior):
 print(os.listdir(prior))

prior exists? False


In [4]:
# Update the script to write to current workspace. Also let's do a quick timing test on one block to estimate the budget.
import time, mpmath as mp
import sys
sys.path.insert(0, '/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21')
import importlib
import ldh_worker
importlib.reload(ldh_worker)

# Quick test: low-t block
t0 = time.time()
res = ldh_worker.find_zeros_in_block_v2((0.0, 25.0, 0.5, 30, 50))
print("low-t block [0,25]:", len(res), "zeros, time=", time.time()-t0, "s")
print("first few:", res[:3])

low-t block [0,25]: 8 zeros, time= 1.590407133102417 s
first few: ['5.0941598445710949256987955170797974750670744531091', '8.9399144080965232447176128773475666256121616213856', '12.133545425800742403702547753765281383126636124036']


In [5]:
# Test a higher-t block to estimate scaling
import time
t0 = time.time()
res = ldh_worker.find_zeros_in_block_v2((1000.0, 1025.0, 0.5, 30, 50))
print("block [1000,1025]:", len(res), "zeros, time=", time.time()-t0, "s")

t0 = time.time()
res = ldh_worker.find_zeros_in_block_v2((2000.0, 2025.0, 0.5, 30, 50))
print("block [2000,2025]:", len(res), "zeros, time=", time.time()-t0, "s")

block [1000,1025]: 21 zeros, time= 32.88014101982117 s


block [2000,2025]: 27 zeros, time= 69.09203052520752 s


In [6]:
# Estimate total time. Need to scan to a t value that yields ~5000 zeros.
# From Riemann-von Mangoldt-type density, L_DH has zero density ~ (t/2π)*log(t) roughly.
# But here at t=1000 we got 21 zeros in 25 units = 0.84/unit. cumulative: integrate.
# 5000 zeros need roughly t~4600 per the script.
# Single-core total time estimate: sum of per-block times.
# At t=1000: 33s/block, at t=2000: 69s/block. Roughly linear in t.
# Per block time ~ 0.034*t + offset. Sum from 0 to 4600 in 25 steps = 184 blocks
# Mean block time ~ 0.034*2300 + small ~ 78s. Total single core = 184*78 = 14,352s
# With 8 workers: ~1800s = 30 min. Plus overhead. Should fit in budget.
# Let me also test high-t block
import time
t0 = time.time()
res = ldh_worker.find_zeros_in_block_v2((4000.0, 4025.0, 0.5, 30, 50))
print("block [4000,4025]:", len(res), "zeros, time=", time.time()-t0, "s")

block [4000,4025]: 24 zeros, time= 106.10063338279724 s


In [7]:
# Refine estimate. Times: t=0: 1.6, t=1000: 33, t=2000: 69, t=4000: 106. Looks ~ linear in t.
# Fit linear: time(t) ≈ a + b*t
import numpy as np
ts = np.array([12.5, 1012.5, 2012.5, 4012.5])
ys = np.array([1.6, 32.9, 69.1, 106.1])
p = np.polyfit(ts, ys, 1)
print("Fit:", p) # slope, intercept

block_centers = np.arange(12.5, 4600, 25.0)
total_serial = float(np.sum(np.polyval(p, block_centers)))
print(f"Total serial time est: {total_serial:.0f}s = {total_serial/60:.1f} min")
print(f"With 8 workers (perfect parallel): {total_serial/8:.0f}s = {total_serial/8/60:.1f} min")
# With overhead and imbalance, likely 1.2-1.5x. Should be ~ 30-50 min. OK.

Fit: [0.02611714 6.39353571]
Total serial time est: 12229s = 203.8 min
With 8 workers (perfect parallel): 1529s = 25.5 min


In [8]:
# Good - feasible. Update run_parallel2.py path to current workspace and execute.
# Modify in-memory & write a fixed version.
script = open('/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21/run_parallel2.py').read()
script_fixed = script.replace(
 '"/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481"',
 '"/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21"'
)
open('/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21/run_parallel2_fixed.py', 'w').write(script_fixed)
print("written")
# Quick check
print([l for l in script_fixed.splitlines() if 'workspace' in l])

written
['sys.path.insert(0, "/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21")', 'partial_file = "/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_partial.pkl"', 'log_file = "/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_run.log"']


In [9]:
script_fixed = script_fixed.replace(
 '"/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_partial.pkl"',
 '"/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21/ldh_partial.pkl"'
).replace(
 '"/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_run.log"',
 '"/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21/ldh_run.log"'
)
open('/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21/run_parallel2_fixed.py', 'w').write(script_fixed)
print([l for l in script_fixed.splitlines() if 'workspace' in l])

['sys.path.insert(0, "/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21")', 'partial_file = "/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21/ldh_partial.pkl"', 'log_file = "/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21/ldh_run.log"']


In [10]:
# Free kernel memory of prior mpmath state by deleting locals; then run the script.
# Important: kernel has an event loop, but we can use subprocess to run the script.
# Better to run via subprocess so we don't block the kernel. Let me launch and monitor.
import subprocess, os, time
log_path = "/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21/ldh_run.log"
if os.path.exists(log_path):
 os.remove(log_path)

# Launch in background
proc = subprocess.Popen(
 ["python", "/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21/run_parallel2_fixed.py"],
 stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
 cwd="/workspace/58ff7dcc-c667-42d6-9d76-4369a9b08f21"
)
print("Launched pid", proc.pid)
start_time = time.time()

Launched pid 133


In [11]:
import time, os
time.sleep(60)
print(f"Elapsed: {time.time()-start_time:.0f}s")
print(f"proc alive: {proc.poll() is None}")
if os.path.exists(log_path):
 with open(log_path) as f:
 lines = f.readlines()
 print(f"Log lines: {len(lines)}")
 print("Last 5:")
 for l in lines[-5:]:
 print(l.rstrip())

Elapsed: 66s
proc alive: True
Log lines: 20
Last 5:
[ 47.6s] block 15/184 [350,375]: 20 z, completed=15, tot=258
[ 53.4s] block 16/184 [375,400]: 23 z, completed=16, tot=281
[ 53.4s] block 17/184 [400,425]: 21 z, completed=17, tot=302
[ 57.8s] block 18/184 [425,450]: 22 z, completed=18, tot=324
[ 59.1s] block 19/184 [450,475]: 21 z, completed=19, tot=345


In [12]:
import time, os
time.sleep(300) # wait 5 more minutes
print(f"Elapsed: {time.time()-start_time:.0f}s")
print(f"proc alive: {proc.poll() is None}")
with open(log_path) as f:
 lines = f.readlines()
print(f"Log lines: {len(lines)}")
for l in lines[-5:]:
 print(l.rstrip())

Elapsed: 373s
proc alive: True
Log lines: 54
[ 318.6s] block 49/184 [1200,1225]: 24 z, completed=49, tot=1035
[ 320.9s] block 50/184 [1225,1250]: 21 z, completed=50, tot=1056
[ 336.7s] block 51/184 [1250,1275]: 22 z, completed=51, tot=1078
[ 342.8s] block 52/184 [1275,1300]: 23 z, completed=52, tot=1101
[ 356.5s] block 53/184 [1300,1325]: 24 z, completed=53, tot=1125


In [13]:
# Roughly: 53/184 blocks in 357s. Block rate ~ 0.148 blocks/s but slowing.
# 1125/5000 zeros. Need much more. Let's track progress vs time better.
import time
time.sleep(600) # wait 10 more minutes
print(f"Elapsed: {time.time()-start_time:.0f}s")
print(f"proc alive: {proc.poll() is None}")
with open(log_path) as f:
 lines = f.readlines()
print(f"Log lines: {len(lines)}")
for l in lines[-8:]:
 print(l.rstrip())

Elapsed: 982s
proc alive: True
Log lines: 96
[ 846.4s] block 88/184 [2175,2200]: 26 z, completed=88, tot=1999
[ 868.0s] block 89/184 [2200,2225]: 27 z, completed=89, tot=2026
[ 868.0s] block 90/184 [2225,2250]: 26 z, completed=90, tot=2052
[ 871.6s] block 91/184 [2250,2275]: 26 z, completed=91, tot=2078
[ 906.1s] block 92/184 [2275,2300]: 26 z, completed=92, tot=2104
[ 925.1s] block 93/184 [2300,2325]: 26 z, completed=93, tot=2130
[ 932.3s] block 94/184 [2325,2350]: 26 z, completed=94, tot=2156
[ 939.8s] block 95/184 [2350,2375]: 26 z, completed=95, tot=2182


In [14]:
# 95/184 blocks in 940s. Total time estimate: scale-up.
# Block 50: 320s, block 95: 940s. So 45 blocks in 620s = 13.8s/block on average mid-run.
# But block times grow with t. Remaining = 89 blocks at higher t. Likely ~ 18-25s/block avg in 8-worker context.
# Estimate remaining: ~ 89 * 22 = ~ 1960s. Total ~ 2900s. Well within budget.
import time
time.sleep(600)
print(f"Elapsed: {time.time()-start_time:.0f}s")
print(f"proc alive: {proc.poll() is None}")
with open(log_path) as f:
 lines = f.readlines()
print(f"Log lines: {len(lines)}")
for l in lines[-5:]:
 print(l.rstrip())

Elapsed: 1592s
proc alive: True
Log lines: 121
[ 1520.6s] block 116/184 [2875,2900]: 29 z, completed=116, tot=2739
[ 1540.1s] block 117/184 [2900,2925]: 29 z, completed=117, tot=2768
[ 1540.1s] block 118/184 [2925,2950]: 24 z, completed=118, tot=2792
[ 1540.1s] block 119/184 [2950,2975]: 21 z, completed=119, tot=2813
[ 1581.3s] block 120/184 [2975,3000]: 22 z, completed=120, tot=2835


In [15]:
# 120/184 in 1581s. 64 blocks remaining. Need to estimate at higher t. 
# block 95-120: 25 blocks in 641s = 25.6s/block
# blocks 120-184 will be slower. Probably 30s/block. 64*30 = 1920s. Total ~ 3500s. Tight but OK.
import time
time.sleep(600)
print(f"Elapsed: {time.time()-start_time:.0f}s")
print(f"proc alive: {proc.poll() is None}")
with open(log_path) as f:
 lines = f.readlines()
print(f"Log lines: {len(lines)}")
for l in lines[-5:]:
 print(l.rstrip())

Elapsed: 2201s
proc alive: True
Log lines: 145
[ 2124.5s] block 140/184 [3475,3500]: 26 z, completed=140, tot=3370
[ 2128.4s] block 141/184 [3500,3525]: 27 z, completed=141, tot=3397
[ 2128.4s] block 142/184 [3525,3550]: 22 z, completed=142, tot=3419
[ 2149.2s] block 143/184 [3550,3575]: 25 z, completed=143, tot=3444
[ 2196.1s] block 144/184 [3575,3600]: 29 z, completed=144, tot=3473


In [16]:
# 144/184 done at 2196s. 40 blocks remain. ~25s/block at high t = 1000s more. Total ~3200s. Within budget.
# But I have 1011s remaining warning. Wait more cautiously.
import time
time.sleep(500)
print(f"Elapsed: {time.time()-start_time:.0f}s")
print(f"proc alive: {proc.poll() is None}")
with open(log_path) as f:
 lines = f.readlines()
print(f"Log lines: {len(lines)}")
for l in lines[-5:]:
 print(l.rstrip())

TimeoutError: Code execution timed out after 402.0 seconds